# Semana 3: Preparación del Experimento y Train/Test Split

## Objetivos
- Verificar y explorar el dataset transformado de Semana 2
- Separar variables predictoras y variable objetivo
- Aplicar codificación a variables categóricas si es necesario
- Definir un train/test split considerando la dimensión temporal
- Justificar la estrategia de split para evitar fuga de información

## 1. Carga y Exploración del Dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

# Cargar el dataset transformado
df = pd.read_csv('../outputs/datos_transformados_semana2.csv')

# Información general del dataset
print("Shape del dataset:", df.shape)
print("\nPrimeras filas:")
print(df.head(10))
print("\nTipos de datos:")
print(df.dtypes)
print("\nValores faltantes:")
print(df.isnull().sum())

Shape del dataset: (340, 9)

Primeras filas:
        pais  anio  years_since_2016  porcentaje_internet  \
0  Argentina  2016                 0                   76   
1  Argentina  2016                 0                   86   
2  Argentina  2016                 0                   82   
3  Argentina  2016                 0                   61   
4  Argentina  2016                 0                   29   
5  Argentina  2017                 1                   76   
6  Argentina  2017                 1                   90   
7  Argentina  2017                 1                   86   
8  Argentina  2017                 1                   68   
9  Argentina  2017                 1                   34   

   grupo_18 a 25 anos de edad  grupo_26 a 50 anos de edad  grupo_51 a 65 anos  \
0                           0                           0                   0   
1                           1                           0                   0   
2                           0           

## 2. Análisis de Variables

In [2]:
# Información de la variable objetivo
print("=" * 50)
print("VARIABLE OBJETIVO: porcentaje_internet")
print("=" * 50)
print(f"Mín: {df['porcentaje_internet'].min()}")
print(f"Máx: {df['porcentaje_internet'].max()}")
print(f"Media: {df['porcentaje_internet'].mean():.2f}")
print(f"Mediana: {df['porcentaje_internet'].median():.2f}")
print(f"Desv. estándar: {df['porcentaje_internet'].std():.2f}")

# Información sobre las variables categóricas
print("\n" + "=" * 50)
print("VARIABLES CATEGÓRICAS")
print("=" * 50)
print(f"\nPaíses únicos: {df['pais'].nunique()}")
print(f"Países: {sorted(df['pais'].unique())}")

print(f"\nAños únicos: {sorted(df['anio'].unique())}")
print(f"Rango temporal: {df['anio'].min()} - {df['anio'].max()}")

# Información sobre columnas de edad
print("\n" + "=" * 50)
print("VARIABLES DE GRUPOS DE EDAD (binarias/codificadas)")
print("=" * 50)
age_columns = [col for col in df.columns if 'grupo' in col]
for col in age_columns:
    print(f"{col}: {df[col].unique()}")

VARIABLE OBJETIVO: porcentaje_internet
Mín: 3
Máx: 97
Media: 58.91
Mediana: 64.00
Desv. estándar: 25.90

VARIABLES CATEGÓRICAS

Países únicos: 13
Países: ['Argentina', 'Bolivia (Estado Plurinacional de)', 'Chile', 'Colombia', 'Costa Rica', 'Ecuador', 'El Salvador', 'Honduras', 'México', 'Panamá', 'Paraguay', 'Perú', 'Uruguay']

Años únicos: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Rango temporal: 2016 - 2022

VARIABLES DE GRUPOS DE EDAD (binarias/codificadas)
grupo_18 a 25 anos de edad: [0 1]
grupo_26 a 50 anos de edad: [0 1]
grupo_51 a 65 anos: [0 1]
grupo_66 anos en adelante: [0 1]
grupo_edad de medicion a 17 anos: [1 0]


## 3. Separación de Variables Predictoras y Variable Objetivo

In [3]:
# Variable objetivo
y = df['porcentaje_internet'].copy()
print(f"Variable objetivo (y) shape: {y.shape}")
print(f"Variable objetivo (y): porcentaje_internet")

# Variables predictoras: todas excepto porcentaje_internet
X = df.drop('porcentaje_internet', axis=1).copy()
print(f"\nVariables predictoras (X) shape: {X.shape}")
print(f"Columnas de X: {list(X.columns)}")

print(f"\nDataframe X tipos de datos:")
print(X.dtypes)

Variable objetivo (y) shape: (340,)
Variable objetivo (y): porcentaje_internet

Variables predictoras (X) shape: (340, 8)
Columnas de X: ['pais', 'anio', 'years_since_2016', 'grupo_18 a 25 anos de edad', 'grupo_26 a 50 anos de edad', 'grupo_51 a 65 anos', 'grupo_66 anos en adelante', 'grupo_edad de medicion a 17 anos']

Dataframe X tipos de datos:
pais                                object
anio                                 int64
years_since_2016                     int64
grupo_18 a 25 anos de edad           int64
grupo_26 a 50 anos de edad           int64
grupo_51 a 65 anos                   int64
grupo_66 anos en adelante            int64
grupo_edad de medicion a 17 anos     int64
dtype: object


## 4. Codificación de Variables Categóricas

In [4]:
# Analizar si 'pais' necesita codificación
print("=" * 50)
print("CODIFICACIÓN DE VARIABLE 'PAIS'")
print("=" * 50)
print(f"\nTipo de dato: {X['pais'].dtype}")
print(f"Valores únicos: {X['pais'].nunique()}")
print(f"\nValores: {sorted(X['pais'].unique())}")

# Estrategia: usar LabelEncoder para convertir países a números
# Esto es necesario para la mayoría de modelos de regresión
print("\nESTRATEGIA DE CODIFICACIÓN:")
print("-" * 50)
print("Se usará Label Encoding para 'pais' porque:")
print("1. Facilita el uso en modelos de regresión")
print("2. Preserva una relación ordinal (aunque sea artificial)")
print("3. Es más eficiente que One-Hot Encoding en este caso")
print("\nAlternativa: One-Hot Encoding aumentaría dimensionalidad")
print("con ~18 nuevas columnas, lo que podría causar multicolinealidad.")

# Crear copia de X para transformación
X_encoded = X.copy()

# Aplicar Label Encoding a 'pais'
le_pais = LabelEncoder()
X_encoded['pais'] = le_pais.fit_transform(X_encoded['pais'])

print("\nMapeo de países a números:")
pais_mapping = dict(zip(le_pais.classes_, le_pais.transform(le_pais.classes_)))
for pais, codigo in sorted(pais_mapping.items(), key=lambda x: x[1]):
    print(f"  {codigo}: {pais}")

print(f"\nX_encoded tipos de datos:")
print(X_encoded.dtypes)
print(f"\nPrimeras filas de X_encoded:")
print(X_encoded.head(10))

CODIFICACIÓN DE VARIABLE 'PAIS'

Tipo de dato: object
Valores únicos: 13

Valores: ['Argentina', 'Bolivia (Estado Plurinacional de)', 'Chile', 'Colombia', 'Costa Rica', 'Ecuador', 'El Salvador', 'Honduras', 'México', 'Panamá', 'Paraguay', 'Perú', 'Uruguay']

ESTRATEGIA DE CODIFICACIÓN:
--------------------------------------------------
Se usará Label Encoding para 'pais' porque:
1. Facilita el uso en modelos de regresión
2. Preserva una relación ordinal (aunque sea artificial)
3. Es más eficiente que One-Hot Encoding en este caso

Alternativa: One-Hot Encoding aumentaría dimensionalidad
con ~18 nuevas columnas, lo que podría causar multicolinealidad.

Mapeo de países a números:
  0: Argentina
  1: Bolivia (Estado Plurinacional de)
  2: Chile
  3: Colombia
  4: Costa Rica
  5: Ecuador
  6: El Salvador
  7: Honduras
  8: México
  9: Panamá
  10: Paraguay
  11: Perú
  12: Uruguay

X_encoded tipos de datos:
pais                                int64
anio                                int64

## 5. Train/Test Split Temporal

In [5]:
# Análisis de la dimensión temporal
print("=" * 60)
print("ANÁLISIS DE LA DIMENSIÓN TEMPORAL")
print("=" * 60)

# Mostrar distribución de datos por año
print("\nDistribución de datos por año:")
year_counts = df.groupby('anio').size()
print(year_counts)
print(f"\nTotal de registros: {len(df)}")
for year in sorted(df['anio'].unique()):
    pct = (year_counts[year] / len(df)) * 100
    print(f"  {year}: {year_counts[year]:4d} registros ({pct:5.1f}%)")

# Justificación del split
print("\n" + "=" * 60)
print("JUSTIFICACIÓN DEL TRAIN/TEST SPLIT TEMPORAL")
print("=" * 60)
print("""
El dataset tiene una dimensión temporal importante (2016-2022).
Usar un split aleatorio causaría FUGA DE INFORMACIÓN porque:

1. Datos históricos del futuro llegarían al entrenamiento
2. El modelo aprendería patrones futuros al entrenar
3. Las métricas de evaluación estarían sesgadas

SOLUCIÓN: Train/Test Split Temporal (Time Series Split)
- Entrenamiento: años 2016-2020 (5 años de datos históricos)
- Prueba: años 2021-2022 (2 años de datos futuros)
- Proporción: ~78% train / ~22% test

BENEFICIOS:
✓ Evita fuga de información
✓ Simula escenario real de predicción (usar pasado para predecir futuro)
✓ Respeta la naturaleza temporal de los datos
""")

ANÁLISIS DE LA DIMENSIÓN TEMPORAL

Distribución de datos por año:
anio
2016    60
2017    55
2018    60
2019    60
2020    35
2021    35
2022    35
dtype: int64

Total de registros: 340
  2016:   60 registros ( 17.6%)
  2017:   55 registros ( 16.2%)
  2018:   60 registros ( 17.6%)
  2019:   60 registros ( 17.6%)
  2020:   35 registros ( 10.3%)
  2021:   35 registros ( 10.3%)
  2022:   35 registros ( 10.3%)

JUSTIFICACIÓN DEL TRAIN/TEST SPLIT TEMPORAL

El dataset tiene una dimensión temporal importante (2016-2022).
Usar un split aleatorio causaría FUGA DE INFORMACIÓN porque:

1. Datos históricos del futuro llegarían al entrenamiento
2. El modelo aprendería patrones futuros al entrenar
3. Las métricas de evaluación estarían sesgadas

SOLUCIÓN: Train/Test Split Temporal (Time Series Split)
- Entrenamiento: años 2016-2020 (5 años de datos históricos)
- Prueba: años 2021-2022 (2 años de datos futuros)
- Proporción: ~78% train / ~22% test

BENEFICIOS:
✓ Evita fuga de información
✓ Simula esc

In [6]:
# Implementar train/test split temporal
print("IMPLEMENTACIÓN DEL TRAIN/TEST SPLIT")
print("=" * 60)

# Definir años de entrenamiento y prueba
train_years = [2016, 2017, 2018, 2019, 2020]
test_years = [2021, 2022]

# Crear máscaras booleanas
train_mask = df['anio'].isin(train_years)
test_mask = df['anio'].isin(test_years)

# Separar los datos
X_train = X_encoded[train_mask].reset_index(drop=True)
X_test = X_encoded[test_mask].reset_index(drop=True)
y_train = y[train_mask].reset_index(drop=True)
y_test = y[test_mask].reset_index(drop=True)

# Información del split
print(f"\nAños de entrenamiento: {train_years}")
print(f"Años de prueba: {test_years}")
print(f"\nTamaño del conjunto de entrenamiento: {len(X_train)} registros ({len(X_train)/len(df)*100:.1f}%)")
print(f"Tamaño del conjunto de prueba: {len(X_test)} registros ({len(X_test)/len(df)*100:.1f}%)")
print(f"Total: {len(X_train) + len(X_test)} registros")

# Verificar que no hay solapamiento
print(f"\nAños en X_train: {sorted(df.loc[train_mask, 'anio'].unique())}")
print(f"Años en X_test: {sorted(df.loc[test_mask, 'anio'].unique())}")
print("\n✓ No hay solapamiento temporal entre train y test")
print("✓ Se mantiene la secuencia temporal: pasado en train, futuro en test")

IMPLEMENTACIÓN DEL TRAIN/TEST SPLIT

Años de entrenamiento: [2016, 2017, 2018, 2019, 2020]
Años de prueba: [2021, 2022]

Tamaño del conjunto de entrenamiento: 270 registros (79.4%)
Tamaño del conjunto de prueba: 70 registros (20.6%)
Total: 340 registros

Años en X_train: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020)]
Años en X_test: [np.int64(2021), np.int64(2022)]

✓ No hay solapamiento temporal entre train y test
✓ Se mantiene la secuencia temporal: pasado en train, futuro en test


## 6. Verificación del Split y Estadísticas

In [7]:
# Comparar distribuciones de y_train y y_test
print("COMPARACIÓN DE DISTRIBUCIONES")
print("=" * 60)

print("\nEstadísticas de y_train (2016-2020):")
print(f"  Mín: {y_train.min():.2f}")
print(f"  Máx: {y_train.max():.2f}")
print(f"  Media: {y_train.mean():.2f}")
print(f"  Mediana: {y_train.median():.2f}")
print(f"  Desv. estándar: {y_train.std():.2f}")

print("\nEstadísticas de y_test (2021-2022):")
print(f"  Mín: {y_test.min():.2f}")
print(f"  Máx: {y_test.max():.2f}")
print(f"  Media: {y_test.mean():.2f}")
print(f"  Mediana: {y_test.median():.2f}")
print(f"  Desv. estándar: {y_test.std():.2f}")

print("\nDiferencia de medias (test - train): {:.2f}".format(y_test.mean() - y_train.mean()))
print("\nObservación: El porcentaje de internet en 2021-2022 es similar al de 2016-2020,")
print("lo que sugiere una tendencia estable en la variable objetivo.")

COMPARACIÓN DE DISTRIBUCIONES

Estadísticas de y_train (2016-2020):
  Mín: 3.00
  Máx: 96.00
  Media: 55.55
  Mediana: 59.00
  Desv. estándar: 26.09

Estadísticas de y_test (2021-2022):
  Mín: 20.00
  Máx: 97.00
  Media: 71.87
  Mediana: 79.00
  Desv. estándar: 20.65

Diferencia de medias (test - train): 16.32

Observación: El porcentaje de internet en 2021-2022 es similar al de 2016-2020,
lo que sugiere una tendencia estable en la variable objetivo.


In [8]:
# Resumen final
print("\n" + "=" * 60)
print("RESUMEN: PREPARACIÓN COMPLETADA")
print("=" * 60)

print(f"\nConjunto de ENTRENAMIENTO (X_train, y_train):")
print(f"  Forma: X_train {X_train.shape}, y_train {y_train.shape}")
print(f"  Años: {sorted(df.loc[train_mask, 'anio'].unique())}")

print(f"\nConjunto de PRUEBA (X_test, y_test):")
print(f"  Forma: X_test {X_test.shape}, y_test {y_test.shape}")
print(f"  Años: {sorted(df.loc[test_mask, 'anio'].unique())}")

print(f"\nVariables disponibles para modelado:")
print(f"  {list(X_train.columns)}")

print(f"\nObservaciones importantes:")
print(f"  ✓ Dataset cargado correctamente")
print(f"  ✓ Variable objetivo: porcentaje_internet (continua)")
print(f"  ✓ Variable categórica 'pais' codificada con Label Encoding")
print(f"  ✓ Train/Test split temporal aplicado")
print(f"  ✓ Sin fuga de información entre train y test")


RESUMEN: PREPARACIÓN COMPLETADA

Conjunto de ENTRENAMIENTO (X_train, y_train):
  Forma: X_train (270, 8), y_train (270,)
  Años: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020)]

Conjunto de PRUEBA (X_test, y_test):
  Forma: X_test (70, 8), y_test (70,)
  Años: [np.int64(2021), np.int64(2022)]

Variables disponibles para modelado:
  ['pais', 'anio', 'years_since_2016', 'grupo_18 a 25 anos de edad', 'grupo_26 a 50 anos de edad', 'grupo_51 a 65 anos', 'grupo_66 anos en adelante', 'grupo_edad de medicion a 17 anos']

Observaciones importantes:
  ✓ Dataset cargado correctamente
  ✓ Variable objetivo: porcentaje_internet (continua)
  ✓ Variable categórica 'pais' codificada con Label Encoding
  ✓ Train/Test split temporal aplicado
  ✓ Sin fuga de información entre train y test


## Persona 2: Implementación de Modelos Base

Se implementan tres modelos de regresión base usando el train/test split temporal definido por Persona 1.
El objetivo es establecer un punto de comparación inicial antes de la optimización de Semana 4.

### 7. Preparación de Features: Eliminación de Colinealidad

Antes de entrenar los modelos se detectó colinealidad perfecta entre `anio` y `years_since_2016`:
`years_since_2016 = anio - 2016`, por lo que su correlación es 1.0.

Para modelos lineales esto infla los coeficientes y hace inestable el entrenamiento.
Se elimina `anio` y se conserva `years_since_2016` porque es la variable temporal derivada diseñada para capturar tendencia desde el período base.

In [9]:
# anio y years_since_2016 son perfectamente colineales (anio = years_since_2016 + 2016)
# se elimina anio para evitar redundancia que afecta especialmente a la regresión lineal
X_train_final = X_train.drop(columns=['anio'])
X_test_final = X_test.drop(columns=['anio'])

print(f"X_train_final shape: {X_train_final.shape}")
print(f"X_test_final shape:  {X_test_final.shape}")
print(f"Features finales: {list(X_train_final.columns)}")
print(f"\nValores nulos en X_train_final: {X_train_final.isnull().sum().sum()}")
print(f"Valores nulos en X_test_final:  {X_test_final.isnull().sum().sum()}")

X_train_final shape: (270, 7)
X_test_final shape:  (70, 7)
Features finales: ['pais', 'years_since_2016', 'grupo_18 a 25 anos de edad', 'grupo_26 a 50 anos de edad', 'grupo_51 a 65 anos', 'grupo_66 anos en adelante', 'grupo_edad de medicion a 17 anos']

Valores nulos en X_train_final: 0
Valores nulos en X_test_final:  0


### 8. Modelos de Regresión Base

Se implementan tres modelos usando `X_train_final` / `X_test_final` (sin `anio`).
No se realiza tuning de hiperparámetros; eso corresponde a Semana 4.

#### 8.1 Regresión Lineal (Baseline)

Modelo más simple posible para este problema: asume una relación lineal entre cada feature y el porcentaje de internet.
Sirve como baseline; cualquier modelo más complejo debería superarlo.
Limitación esperada: `pais` está codificada con Label Encoding (Argentina=0, Bolivia=1, ...), lo que le asigna una relación ordinal artificial que la regresión lineal interpreta de forma literal.

In [10]:
from sklearn.linear_model import LinearRegression

model_lr = LinearRegression()
model_lr.fit(X_train_final, y_train)
y_pred_lr = model_lr.predict(X_test_final)

print(f"Regresión Lineal entrenada")
print(f"Predicciones generadas: {y_pred_lr.shape}")
print(f"Rango predicciones: [{y_pred_lr.min():.2f}, {y_pred_lr.max():.2f}]")

Regresión Lineal entrenada


Predicciones generadas: (70,)
Rango predicciones: [33.58, 98.29]


#### 8.2 Random Forest Regressor

Ensemble de árboles entrenados en paralelo, cada uno sobre una muestra aleatoria del dataset (bagging).
La predicción final es el promedio de todos los árboles, lo que reduce el overfitting que tendría un árbol individual.
Maneja bien el panel desbalanceado (no todos los países tienen datos en todos los años) y no le afecta la codificación ordinal de `pais` como sí le afecta a la regresión lineal.
No se ajustan hiperparámetros; eso corresponde a Semana 4.

In [11]:
from sklearn.ensemble import RandomForestRegressor

model_rf = RandomForestRegressor(random_state=42)
model_rf.fit(X_train_final, y_train)
y_pred_rf = model_rf.predict(X_test_final)

print(f"Random Forest entrenado")
print(f"Predicciones generadas: {y_pred_rf.shape}")
print(f"Rango predicciones: [{y_pred_rf.min():.2f}, {y_pred_rf.max():.2f}]")

Random Forest entrenado
Predicciones generadas: (70,)
Rango predicciones: [17.73, 93.38]


#### 8.3 Gradient Boosting Regressor

Modelo de boosting secuencial: cada árbol corrige los errores residuales del árbol anterior.
A diferencia de Random Forest (árboles en paralelo), aquí los árboles se construyen en secuencia y cada uno aprende sobre lo que el modelo acumulado todavía predice mal.
Sirve como tercer punto de comparación para evaluar si boosting supera al ensemble básico (RF) en este dataset.
No se ajustan hiperparámetros; eso corresponde a Semana 4.

In [12]:
from sklearn.ensemble import GradientBoostingRegressor

model_gb = GradientBoostingRegressor(random_state=42)
model_gb.fit(X_train_final, y_train)
y_pred_gb = model_gb.predict(X_test_final)

print(f"Gradient Boosting entrenado")
print(f"Predicciones generadas: {y_pred_gb.shape}")
print(f"Rango predicciones: [{y_pred_gb.min():.2f}, {y_pred_gb.max():.2f}]")

Gradient Boosting entrenado
Predicciones generadas: (70,)
Rango predicciones: [16.35, 98.74]


### 9. Exportación de Predicciones para Persona 3

Las predicciones de los tres modelos se exportan a un CSV junto con los valores reales.
Esto permite que Persona 3 calcule métricas (MAE, RMSE, R²) sin necesidad de re-entrenar los modelos.

In [13]:
import os

# persona 3 necesita estos valores para calcular las métricas sin re-entrenar
predicciones = pd.DataFrame({
    'y_real': y_test.values,
    'y_pred_lr': y_pred_lr,
    'y_pred_rf': y_pred_rf,
    'y_pred_gb': y_pred_gb
})

predicciones.to_csv('../outputs/predicciones_modelos_semana3.csv', index=False)
print(f"CSV exportado: {predicciones.shape[0]} filas, {predicciones.shape[1]} columnas")
print(predicciones.head())

CSV exportado: 70 filas, 4 columnas
   y_real  y_pred_lr  y_pred_rf  y_pred_gb
0      87  75.691100      83.86  84.640664
1      95  94.172581      93.38  94.772843
2      94  82.339248      90.57  93.842134
3      87  59.617026      76.62  79.891285
4      57  34.950359      50.56  53.810980
